In [3]:
!pip install -q transformers accelerate peft bitsandbytes datasets trl


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [4]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)

from peft import LoraConfig, get_peft_model
from datasets import load_dataset


print("CUDA:", torch.cuda.is_available())
print("GPU :", torch.cuda.get_device_name(0))

CUDA: False


RuntimeError: Found no NVIDIA driver on your system. Please check that you have an NVIDIA GPU and installed a driver from http://www.nvidia.com/Download/index.aspx

In [5]:
model_name          = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer           = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
print("tokenizer ready")

tokenizer ready


In [6]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit             = True,
    bnb_4bit_quant_type      = "nf4",
    bnb_4bit_compute_dtype   = torch.float16,
    bnb_4bit_use_double_quant= True
)

model                  = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config = bnb_config,
    device_map          = "auto"
)
model.config.use_cache = False
print("model loaded")
print("memory used:", round(torch.cuda.memory_allocated() / 1e9, 2), "GB")

Loading weights: 100%|██████████| 201/201 [00:22<00:00,  8.84it/s]


model loaded
memory used: 0.0 GB


In [5]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "v_proj"
    ]
)

model = get_peft_model(
    model,
    lora_config
)

model.print_trainable_parameters()

trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.2044


In [6]:
from google.colab import files
uploaded = files.upload()

Saving val.jsonl to val.jsonl
Saving train.jsonl to train.jsonl


In [7]:
dataset = load_dataset(
    "json",
    data_files={
        "train"     : "/content/train.jsonl",
        "validation": "/content/val.jsonl"
    }
)

def format_prompt(example):
    return {
        "text": f"### Instruction:\n{example['instruction']}\n\n### Input:\n{example['input']}\n\n### Response:\n{example['output']}"
    }

def tokenize(example):
    return tokenizer(
        format_prompt(example)["text"],
        truncation = True,
        max_length = 256,
        padding    = "max_length"
    )

tokenized = dataset.map(tokenize, remove_columns=dataset["train"].column_names)
print(tokenized)

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/1282 [00:00<?, ? examples/s]

Map:   0%|          | 0/143 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 1282
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 143
    })
})


In [ ]:
training_args = TrainingArguments(
    output_dir="/content/lora_outputs",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=2e-4,
    num_train_epochs=3,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    fp16=True,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    report_to="none"
)

print("training args ready")

training args ready


In [9]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=data_collator
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,0.959320,0.983969
2,0.974849,0.964899
3,0.847569,0.960038


TrainOutput(global_step=963, training_loss=0.963768369807385, metrics={'train_runtime': 736.5651, 'train_samples_per_second': 5.222, 'train_steps_per_second': 1.307, 'total_flos': 6124644706811904.0, 'train_loss': 0.963768369807385, 'epoch': 3.0})

In [10]:
adapter_path = "/content/adapters"

trainer.model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

print("adapters saved")

adapters saved
